# English Premier League (EPL) Match Outcome Prediction

This notebook demonstrates an end-to-end machine learning workflow to predict English Premier League match outcomes (specifically, home wins) using historical match data and rolling team form statistics.

In [147]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, classification_report, confusion_matrix

## 1. Data Loading and Filtering

We start by importing the essential libraries, loading the complete dataset containing historical EPL match statistics, and filtering it for recent seasons.

In [148]:
# Load dataset and filter for recent seasons (Season 2223 onwards)
matches = pd.read_csv('/content/sample_data/EPL_Match_Data_2000_2026.csv')
matches_recent = matches[matches['Season'] >= 2223].copy()
print(f"Loaded {len(matches_recent)} records from recent seasons.")
display(matches_recent.head())

Loaded 1520 records from recent seasons.


,Season,Year,Month,Day,WeekDay,Time,Round,HomeTeam,AwayTeam,HomeGoal,...,EuropaLeagueHome,EuropaLeagueAway,ConferenceLeagueHome,ConferenceLeagueAway,SuperCupHome,SuperCupAway,ClubWorldCupHome,ClubWorldCupAway,IntercontinentalCupHome,IntercontinentalCupAway
8360,2223,2022,8,5,Sun,16:30,17.0,Crystal Palace,Arsenal,0,...,No,No,No,No,No,No,No,No,No,No
8361,2223,2022,8,6,Sun,16:30,17.0,Tottenham,Southampton,4,...,No,No,No,No,No,No,No,No,No,No
8362,2223,2022,8,6,Mon,20:00,17.0,Everton,Chelsea,0,...,No,No,No,No,No,No,No,No,No,No
8363,2223,2022,8,6,Mon,20:00,17.0,Newcastle,Nottingham,2,...,No,No,No,No,No,No,No,No,No,No
8364,2223,2022,8,6,Mon,20:00,17.0,Fulham,Liverpool,2,...,No,No,No,No,No,No,No,No,No,No


In [149]:
matches.head()

,Season,Year,Month,Day,WeekDay,Time,Round,HomeTeam,AwayTeam,HomeGoal,...,EuropaLeagueHome,EuropaLeagueAway,ConferenceLeagueHome,ConferenceLeagueAway,SuperCupHome,SuperCupAway,ClubWorldCupHome,ClubWorldCupAway,IntercontinentalCupHome,IntercontinentalCupAway
0,1,2000,8,19,Sat,15:00,1.0,Charlton,Man City,4,...,No,No,No,No,No,No,No,No,No,No
1,1,2000,8,19,Sat,15:00,1.0,Leeds,Everton,2,...,No,No,No,No,No,No,No,No,No,No
2,1,2000,8,19,Sat,15:00,1.0,Derby,Southampton,2,...,No,No,No,No,No,No,No,No,No,No
3,1,2000,8,19,Sat,15:00,1.0,Chelsea,West Ham,4,...,No,No,No,No,No,No,No,No,No,No
4,1,2000,8,19,Sat,15:00,1.0,Tottenham,Ipswich,3,...,No,No,No,No,No,No,No,No,No,No


In [150]:
matches_recent = matches[matches['Season'] >= 2223]

## 2. Feature Engineering & Preprocessing

Next, we construct baseline features, categorize team codes, parse timestamps, and define our target prediction metric (a Home Win).

In [151]:
matches_recent.columns.tolist()

['Season',
 'Year',
 'Month',
 'Day',
 'WeekDay',
 'Time',
 'Round',
 'HomeTeam',
 'AwayTeam',
 'HomeGoal',
 'AwayGoal',
 'Result',
 'HomeGoal-HalfTime',
 'AwayGoal-HalfTime',
 'Result-HalfTime',
 'Attendance',
 'Referee',
 'HomeShot',
 'AwayShot',
 'HomeOnTargetShot',
 'AwayOnTargetShot',
 'HomeFoul',
 'AwayFoul',
 'HomeCorner',
 'AwayCorner',
 'HomeYellow',
 'AwayYellow',
 'HomeRed',
 'AwayRed',
 'XGHome',
 'XGAway',
 'SeasonPosHome',
 'SeasonPlayedHome',
 'SeasonWinHome',
 'SeasonDrawHome',
 'SeasonLossHome',
 'SeasonGFHome',
 'SeasonGAHome',
 'SeasonGDHome',
 'SeasonPointsHome',
 'QualificationOrRelegationHome',
 'SeasonPosAway',
 'SeasonPlayedAway',
 'SeasonWinAway',
 'SeasonDrawAway',
 'SeasonLossAway',
 'SeasonGFAway',
 'SeasonGAAway',
 'SeasonGDAway',
 'SeasonPointsAway',
 'QualificationOrRelegationAway',
 'HomeCity',
 'HomeRegion',
 'AwayCity',
 'AwayRegion',
 'IsDerby',
 'HomeStadium',
 'HomeManager',
 'AwayManager',
 'HomeTrophies',
 'AwayTrophies',
 'HomeForm',
 'AwayForm',

In [152]:
# 1. Map weekdays to numeric codes
day_map = {'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6}
matches_recent['DayCode'] = matches_recent['WeekDay'].map(day_map)

# 2. Categorize Home and Away Team codes
matches_recent['HomeCode'] = matches_recent['HomeTeam'].astype('category').cat.codes
matches_recent['AwayCode'] = matches_recent['AwayTeam'].astype('category').cat.codes

# 3. Format game kick-off hour
matches_recent['Hour'] = matches_recent['Time'].str.replace(":", "").str.slice(0, 2).astype("int")

# 4. Set binary Target (1 if Home Team Wins, else 0)
matches_recent['Target'] = (matches_recent['Result'] == 'H').astype('int')

print("Baseline features built successfully.")

Baseline features built successfully.


In [153]:
matches_recent['HomeCode'] = matches_recent['HomeTeam'].astype('category').cat.codes
matches_recent['AwayCode'] = matches_recent['AwayTeam'].astype('category').cat.codes

In [154]:
matches_recent['Hour'] = matches_recent['Time'].str.replace(":","").str.slice(0,2).astype("int")

In [155]:
matches_recent['Target'] = (matches_recent['Result'] == 'H').astype('int')

## 3. Rolling Form Statistics (Home & Away Teams)

Form is a crucial predictor in sports forecasting. We compute 3-match rolling averages of performance indicators (shots, goals, corners, xG) for both home and away teams to capture recent momentum.

In [156]:
from sklearn.ensemble import RandomForestClassifier

In [157]:
rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)

In [158]:
predictors = ['HomeCode', 'AwayCode', 'DayCode', 'Hour']

In [159]:
from sklearn.model_selection import train_test_split
train ,test = train_test_split(matches_recent, test_size=0.20, shuffle=True)

In [160]:
rf.fit(train[predictors], train['Target'])

RandomForestClassifier(min_samples_split=10, n_estimators=50, random_state=1)

In [161]:
preds = rf.predict(test[predictors])

In [162]:
from sklearn.metrics import accuracy_score
accuracy_score(test['Target'], preds)

0.6151315789473685

In [163]:
combined = pd.DataFrame(dict(actual=test['Target'], predicted=preds))

In [164]:
pd.crosstab(index=combined['actual'], columns=combined['predicted'])

predicted,0,1
actual,,
0,114,57
1,60,73


In [165]:
from sklearn.metrics import precision_score

precision_score(test["Target"], preds)

0.5615384615384615

In [166]:
grouped_matches = matches_recent.groupby("HomeTeam")

In [167]:
group = grouped_matches.get_group("Man City")
group

,Season,Year,Month,Day,WeekDay,Time,Round,HomeTeam,AwayTeam,HomeGoal,...,SuperCupAway,ClubWorldCupHome,ClubWorldCupAway,IntercontinentalCupHome,IntercontinentalCupAway,DayCode,HomeCode,AwayCode,Hour,Target
8372,2223,2022,8,13,Tue,20:15,1.0,Man City,Bournemouth,4,...,No,No,No,No,No,1,15,2,20,1
8395,2223,2022,8,27,Sat,17:30,19.0,Man City,Crystal Palace,4,...,No,No,No,No,No,5,15,7,17,1
8406,2223,2022,8,31,Mon,20:00,19.0,Man City,Nottingham,6,...,No,No,No,No,No,0,15,18,20,1
8434,2223,2022,10,2,Wed,18:00,20.0,Man City,Man United,6,...,No,No,No,No,No,2,15,16,18,1
8440,2223,2022,10,8,Wed,18:00,20.0,Man City,Southampton,4,...,No,No,No,No,No,2,15,20,18,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9788,2526,2026,3,4,Sat,15:00,22.0,Man City,Nottingham,2,...,No,No,No,No,No,5,15,18,15,0
9825,2526,2026,4,19,Sat,19:45,25.0,Man City,Arsenal,2,...,No,No,No,No,No,5,15,0,19,1
9853,2526,2026,5,9,Sat,15:00,27.0,Man City,Brentford,3,...,No,No,No,No,No,5,15,3,15,1
9859,2526,2026,5,13,Sat,15:00,27.0,Man City,Crystal Palace,3,...,No,No,No,No,No,5,15,7,15,1


In [168]:
# Setup performance metrics we want rolling averages for
home_cols = ["XGHome", "SeasonGFHome", "SeasonWinHome", "HomeCorner", "HomeGoal", "HomeShot", "SeasonGAHome", "HomeOnTargetShot", "HomeFoul"]
new_home_cols = [f"{c}_rolling" for c in home_cols]

away_cols = ["XGAway", "SeasonGFAway", "SeasonWinAway", "AwayCorner", "AwayGoal", "AwayShot", "SeasonGAAway", "AwayOnTargetShot", "AwayFoul"]
new_away_cols = [f"{c}_rolling" for c in away_cols]

def calculate_rolling_averages(group, cols, new_cols):
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    return group.dropna(subset=new_cols)

# Apply rolling averages for Home and Away Team performances independently
matches_rolling_home = matches_recent.groupby("HomeTeam", group_keys=False).apply(lambda x: calculate_rolling_averages(x, home_cols, new_home_cols))
matches_rolling_away = matches_recent.groupby("AwayTeam", group_keys=False).apply(lambda x: calculate_rolling_averages(x, away_cols, new_away_cols))

# Reset indices to cleanly merge statistics
matches_rolling_home.index = range(matches_rolling_home.shape[0])
matches_rolling_away.index = range(matches_rolling_away.shape[0])

# Merge computed metrics on match unique identifiers
merge_keys = ['Season', 'Year', 'Month', 'Day', 'WeekDay', 'Time', 'Round', 'HomeTeam', 'AwayTeam']
matches_combined = pd.merge(matches_rolling_home, matches_rolling_away[merge_keys + new_away_cols], on=merge_keys, how='inner')
print(f"Rolling team form features computed. Dataset size: {matches_combined.shape}")

Rolling team form features computed. Dataset size: (1431, 108)


In [169]:
cols = ["XGHome", "SeasonGFHome", "SeasonWinHome", "HomeCorner", "HomeGoal", "HomeShot", "SeasonGAHome", "HomeOnTargetShot", "HomeFoul"]
new_cols = [f"{c}_rolling" for c in cols]

rolling_averages(group, cols, new_cols)

,Season,Year,Month,Day,WeekDay,Time,Round,HomeTeam,AwayTeam,HomeGoal,...,Target,XGHome_rolling,SeasonGFHome_rolling,SeasonWinHome_rolling,HomeCorner_rolling,HomeGoal_rolling,HomeShot_rolling,SeasonGAHome_rolling,HomeOnTargetShot_rolling,HomeFoul_rolling
8434,2223,2022,10,2,Wed,18:00,20.0,Man City,Man United,6,...,1,1.000000,94.0,28.0,9.000000,4.666667,18.000000,33.0,7.000000,10.666667
8440,2223,2022,10,8,Wed,18:00,20.0,Man City,Southampton,4,...,1,1.333333,94.0,28.0,7.000000,5.333333,19.000000,33.0,8.000000,8.666667
8468,2223,2022,10,22,Sun,12:00,21.0,Man City,Brighton,3,...,1,1.033333,94.0,28.0,8.000000,5.333333,20.000000,33.0,9.000000,5.333333
8487,2223,2022,11,5,Tue,20:15,22.0,Man City,Fulham,2,...,1,1.366667,94.0,28.0,7.000000,4.333333,17.666667,33.0,7.333333,6.000000
8500,2223,2022,11,12,Wed,19:30,22.0,Man City,Brentford,1,...,0,1.266667,94.0,28.0,8.333333,3.000000,15.666667,33.0,5.666667,7.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9788,2526,2026,3,4,Sat,15:00,22.0,Man City,Nottingham,2,...,0,0.933333,77.0,23.0,4.333333,2.333333,13.000000,35.0,5.333333,12.000000
9825,2526,2026,4,19,Sat,19:45,25.0,Man City,Arsenal,2,...,1,1.400000,77.0,23.0,5.000000,2.333333,16.333333,35.0,6.333333,10.000000
9853,2526,2026,5,9,Sat,15:00,27.0,Man City,Brentford,3,...,1,1.500000,77.0,23.0,6.333333,2.000000,17.000000,35.0,6.333333,9.000000
9859,2526,2026,5,13,Sat,15:00,27.0,Man City,Crystal Palace,3,...,1,1.466667,77.0,23.0,8.000000,2.333333,20.333333,35.0,7.333333,7.666667


In [170]:
matches_rolling = matches_recent.groupby("HomeTeam").apply(lambda x: rolling_averages(x, cols, new_cols))

In [171]:
matches_rolling

Season  Year  Month  Day WeekDay   Time  Round HomeTeam  \
HomeTeam                                                                 
Arsenal  8430    2223  2022     10    1     Tue  18:00   20.0  Arsenal   
         8443    2223  2022     10    9     Wed  19:30   20.0  Arsenal   
         8485    2223  2022     10   30     Tue  20:15   22.0  Arsenal   
         8506    2223  2022     12   26     Sat  12:30   23.0  Arsenal   
         8527    2223  2023      1    3     Sun  14:00   23.0  Arsenal   
...               ...   ...    ...  ...     ...    ...    ...      ...   
Wolves   9771    2526  2026      2   27     Sat  12:30   21.0   Wolves   
         9784    2526  2026      3    3     Sat  15:00   22.0   Wolves   
         9836    2526  2026      4   25     Wed  20:00    7.0   Wolves   
         9841    2526  2026      5    2     Sat  15:00   26.0   Wolves   
         9862    2526  2026      5   17     Sun  14:00   27.0   Wolves   

                  AwayTeam  HomeGoal  ...  Target XGHome_rolling  \
HomeTeam                              ...                          
Arsenal  8430    Tottenham         3  ...       1       1.266667   
         8443    Liverpool         3  ...       1       0.833333   
         8485   Nottingham         5  ...       1       0.766667   
         8506     West Ham         3  ...       1       1.966667   
         8527    Newcastle         0  ...       0       2.133333   
...                    ...       ...  ...     ...            ...   
Wolves   9771  Aston Villa         2  ...       1       1.366667   
         9784    Liverpool         2  ...       1       1.600000   
         9836    Tottenham         0  ...       0       1.366667   
         9841   Sunderland         1  ...       0       1.033333   
         9862       Fulham         1  ...       0       0.933333   

               SeasonGFHome_rolling  SeasonWinHome_rolling HomeCorner_rolling  \
HomeTeam                                                                        
Arsenal  8430                  88.0                   26.0           8.333333   
         8443                  88.0                   26.0           8.000000   
         8485                  88.0                   26.0           6.333333   
         8506                  88.0                   26.0           6.000000   
         8527                  88.0                   26.0           6.666667   
...                             ...                    ...                ...   
Wolves   9771                  27.0                    3.0           3.000000   
         9784                  27.0                    3.0           1.333333   
         9836                  27.0                    3.0           1.000000   
         9841                  27.0                    3.0           2.333333   
         9862                  27.0                    3.0           4.333333   

               HomeGoal_rolling HomeShot_rolling  SeasonGAHome_rolling  \
HomeTeam                                                                 
Arsenal  8430          2.666667        21.000000                  43.0   
         8443          2.333333        22.000000                  43.0   
         8485          2.666667        18.333333                  43.0   
         8506          3.666667        19.000000                  43.0   
         8527          3.666667        17.000000                  43.0   
...                         ...              ...                   ...   
Wolves   9771          1.000000        11.000000                  68.0   
         9784          1.666667         8.333333                  68.0   
         9836          2.000000         6.000000                  68.0   
         9841          1.333333         8.000000                  68.0   
         9862          1.000000        11.666667                  68.0   

               HomeOnTargetShot_rolling  HomeFoul_rolling  
HomeTeam                                                   
Arsenal  8430                 

In [172]:
matches_rolling.index = range(matches_rolling.shape[0])

In [173]:
# Define columns to calculate rolling statistics for the Away Team
away_cols = ["XGAway", "SeasonGFAway", "SeasonWinAway", "AwayCorner", "AwayGoal", "AwayShot", "SeasonGAAway", "AwayOnTargetShot", "AwayFoul"]
new_away_cols = [f"{c}_rolling" for c in away_cols]

def rolling_averages_away(group, cols, new_cols):
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset=new_cols)
    return group

# Compute Away rolling averages grouped by AwayTeam
matches_rolling_away = matches_recent.groupby("AwayTeam").apply(lambda x: rolling_averages_away(x, away_cols, new_away_cols))
matches_rolling_away.index = range(matches_rolling_away.shape[0])


In [174]:
# Merge Home and Away rolling statistics on the match level
# Since each match is unique, we merge on key columns that identify the match uniquely.
merge_keys = ['Season', 'Year', 'Month', 'Day', 'WeekDay', 'Time', 'Round', 'HomeTeam', 'AwayTeam']

matches_combined = pd.merge(
    matches_rolling,
    matches_rolling_away[merge_keys + new_away_cols],
    on=merge_keys,
    how='inner'
)


## 4. Chronological Machine Learning Model & Evaluation

Finally, we define a robust prediction pipeline. Using a `RandomForestClassifier`, we train on past match data and evaluate on subsequent fixtures (chronological train/test split) to strictly prevent any temporal data leakage.

In [175]:
# Define feature list
base_predictors = ['HomeCode', 'AwayCode', 'DayCode', 'Hour']
upgraded_predictors = base_predictors + new_home_cols + new_away_cols

def run_pipeline(data, predictors):
    # Sort chronologically to prevent temporal data leakage
    data_sorted = data.sort_values(['Season', 'Month', 'Day'])

    # Chronological Split (Train: 80%, Test: 20%)
    train, test = train_test_split(data_sorted, test_size=0.20, shuffle=False)

    # Model Initialization & Training
    rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)
    rf.fit(train[predictors], train["Target"])

    # Prediction & Metric Computation
    preds = rf.predict(test[predictors])
    combined = pd.DataFrame(dict(actual=test["Target"], predicted=preds), index=test.index)

    acc = accuracy_score(test["Target"], preds)
    prec = precision_score(test["Target"], preds)

    print("=== Performance Report ===")
    print(f"Accuracy Score  : {acc:.4f}")
    print(f"Precision Score : {prec:.4f}\n")
    print("Confusion Matrix:")
    print(pd.crosstab(index=combined['actual'], columns=combined['predicted']))
    return combined

combined_results = run_pipeline(matches_combined, upgraded_predictors)

=== Performance Report ===
Accuracy Score  : 0.6516
Precision Score : 0.6598

Confusion Matrix:
predicted    0   1
actual            
0          123  33
1           67  64
